# FishScan - training in Google Colab

Before running anything: **Runtime > Change runtime type > T4 GPU > Save**.

Your Google Drive must contain:
```
MyDrive/FishScan/
  src/            (dataset_loader.py, fish_model.py, model_evaluator.py, __init__.py)
  scripts/        (train.py, compare_experiments.py, ...)
  data/ffe_split.csv
  data/xzyx7pbr3w-1.zip   (the FFE zip from Mendeley)
  models/freshness_model.keras + freshness_meta.joblib   (baseline, for the comparison)
  requirements.txt
```
Every new session: run Steps 1-4 first (Step 3 may ask to restart; then continue at Step 4).
Results are saved straight into `MyDrive/FishScan/models` and `MyDrive/FishScan/reports`.

In [1]:
# Step 1: check that a GPU is connected (you should see 'Tesla T4')
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-36406dac-cd7b-c82e-d1d9-01358dd1235b)


In [2]:
# Step 2: connect Google Drive (a pop-up asks you to allow access)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Step 3: install the SAME library versions as the laptop
%cd /content/drive/MyDrive/FishScan
!pip install -q -r requirements.txt
import tensorflow as tf, keras
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('Expected:  TensorFlow 2.21.0 | Keras 3.13.2')
# A pandas warning about google-colab is harmless.
# If Colab offers 'Restart session': click it, then continue with Step 4.

/content/drive/MyDrive/FishScan
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
TensorFlow 2.21.0 | Keras 3.13.2
Expected:  TensorFlow 2.21.0 | Keras 3.13.2


In [4]:
# Step 4: unzip the images to Colab's fast local disk (reading from Drive is slow)
!unzip -q -o /content/drive/MyDrive/FishScan/data/xzyx7pbr3w-1.zip -d /content/ffe
!find /content/ffe -name '*.jpg' | wc -l   # should print 4390
%env FISHSCAN_DATA_DIR=/content/ffe

4390
env: FISHSCAN_DATA_DIR=/content/ffe


In [5]:
# Step 5: train the NORMAL models (already done - only re-run if needed).
#   3 (Both normal models) then 1 (Full training)
%cd /content/drive/MyDrive/FishScan
%run scripts/train.py

/content/drive/MyDrive/FishScan
TensorFlow 2.21.0 | GPU: YES - /physical_device:GPU:0
Images folder: /content/ffe
Split file:    /content/drive/MyDrive/FishScan/data/ffe_split.csv

What do you want to train?
  1. Species model
  2. Freshness model (baseline)
  3. Both normal models
  4. Freshness experiment(s)
Your choice: 4

Experiments:
  - E1_species_aware: 21 classes (species x level), freshness = sum of probabilities; same training settings as the baseline
  - E2_deep_strong_aug: 3 classes; unfreeze 60 layers, up to 20 fine-tuning epochs, patience 4, strong lighting/colour augmentation
  - E3_species_aware_deep_strong: E1's 21 classes + E2's settings (run ONLY if E1 and E2 both beat the baseline)

Which experiment(s)?
  1. E1 then E2 (back to back)
  2. E1 only
  3. E2 only
  4. E3 only
Your choice: 1

Full training or a quick test run?
  1. Full training
  2. Quick test (checks that everything works)
Your choice: 2

========== Training: E1_species_aware (task: species_freshness) 

3/3 - 20s - 7s/step - accuracy: 0.4167 - loss: 1.3587 - val_accuracy: 0.2500 - val_loss: 1.4277

--- Stage 2: fine-tuning the last 60 MobileNetV2 layers ---


3/3 - 23s - 8s/step - accuracy: 0.3333 - loss: 1.4254 - val_accuracy: 0.3438 - val_loss: 1.2521

Validation accuracy (freshness, 3 classes): 34.4%  (loss 1.252)
Saved the model to /content/drive/MyDrive/FishScan/models/quick_test/E2_deep_strong_aug
Saved E2_deep_strong_aug_history.csv to /content/drive/MyDrive/FishScan/models/quick_test/E2_deep_strong_aug

Done.


In [6]:
# Step 5b: freshness EXPERIMENTS (saved under models/experiments/, baseline untouched)
#   First a quick check:  4 (Freshness experiment) -> 1 (E1 then E2) -> 2 (Quick test)
#   Then the real run:    4 (Freshness experiment) -> 1 (E1 then E2) -> 1 (Full training)
#   E3 later ONLY if Step 5c says so:  4 -> 4 (E3 only) -> 1 (Full training)
%cd /content/drive/MyDrive/FishScan
%run scripts/train.py

/content/drive/MyDrive/FishScan
TensorFlow 2.21.0 | GPU: YES - /physical_device:GPU:0
Images folder: /content/ffe
Split file:    /content/drive/MyDrive/FishScan/data/ffe_split.csv

What do you want to train?
  1. Species model
  2. Freshness model (baseline)
  3. Both normal models
  4. Freshness experiment(s)
Your choice: 4

Experiments:
  - E1_species_aware: 21 classes (species x level), freshness = sum of probabilities; same training settings as the baseline
  - E2_deep_strong_aug: 3 classes; unfreeze 60 layers, up to 20 fine-tuning epochs, patience 4, strong lighting/colour augmentation
  - E3_species_aware_deep_strong: E1's 21 classes + E2's settings (run ONLY if E1 and E2 both beat the baseline)

Which experiment(s)?
  1. E1 then E2 (back to back)
  2. E1 only
  3. E2 only
  4. E3 only
Your choice: 1

Full training or a quick test run?
  1. Full training
  2. Quick test (checks that everything works)
Your choice: 1

========== Training: E1_species_aware (task: species_freshness) 

In [7]:
# Step 5c: compare baseline vs experiments on the VALIDATION set only (no test set)
%cd /content/drive/MyDrive/FishScan
%run scripts/compare_experiments.py

/content/drive/MyDrive/FishScan
Scoring Baseline ...
Scoring E1_species_aware ...
Scoring E2_deep_strong_aug ...

Freshness on the VALIDATION set (613 images):
                   val_accuracy val_macro_f1 val_log_loss val_ece share_above_60 accuracy_above_60
Baseline                  57.3%        57.0%        0.895   0.053          48.3%             68.6%
E1_species_aware          59.7%        59.0%        0.933   0.119          69.7%             66.3%
E2_deep_strong_aug        63.3%        63.2%        0.822   0.086          66.2%             72.7%

Decision rule: at least +3% accuracy and ECE not worse than baseline + 0.01
  E1_species_aware: +2.4 points, ECE 0.119 -> does not beat the baseline
  E2_deep_strong_aug: +6.0 points, ECE 0.086 -> does not beat the baseline

E3: skip it - E1 and E2 did not both beat the baseline.
Best candidate so far: Baseline (keep the current freshness model).

Saved the table to /content/drive/MyDrive/FishScan/reports/experiments/comparison_val.csv


In [8]:
# Step 6: check the saved files (these are already in your Google Drive)
!ls -lh /content/drive/MyDrive/FishScan/models /content/drive/MyDrive/FishScan/reports
!ls -lhR /content/drive/MyDrive/FishScan/models/experiments /content/drive/MyDrive/FishScan/reports/experiments

/content/drive/MyDrive/FishScan/models:
total 42M
drwx------ 2 root root 4.0K Nov 16 03:46 checkpoints
drwx------ 4 root root 4.0K Nov 16 05:39 experiments
-rw------- 1 root root  103 Nov 16 04:06 freshness_meta.joblib
-rw------- 1 root root  21M Nov 16 04:06 freshness_model.keras
drwx------ 4 root root 4.0K Nov 16 05:23 quick_test
-rw------- 1 root root  190 Nov 16 03:57 species_meta.joblib
-rw------- 1 root root  21M Nov 16 03:57 species_model.keras

/content/drive/MyDrive/FishScan/reports:
total 7.0K
drwx------ 2 root root 4.0K Nov 16 06:11 experiments
-rw------- 1 root root 1.4K Nov 16 04:06 freshness_history.csv
-rw------- 1 root root 1.5K Nov 16 03:57 species_history.csv
/content/drive/MyDrive/FishScan/models/experiments:
total 8.0K
drwx------ 3 root root 4.0K Nov 16 05:39 E1_species_aware
drwx------ 3 root root 4.0K Nov 16 06:08 E2_deep_strong_aug

/content/drive/MyDrive/FishScan/models/experiments/E1_species_aware:
total 22M
drwx------ 2 root root 4.0K Nov 16 05:38 checkpoints
